In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

In [ ]:
import importlib.util
_real_find_spec = getattr(importlib.util, '_real_find_spec', importlib.util.find_spec)
importlib.util._real_find_spec = _real_find_spec
def _no_socksio(name, *a, **kw):
    if name == "socksio": return None
    return _real_find_spec(name, *a, **kw)
importlib.util.find_spec = _no_socksio

import datetime, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import psycopg2
import pytz

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.IRSwapsTB import IRSwapsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from SDRUtils._swappulse_scripts.ingest_usdswaps_tape import resolve_pg_url

warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
NY = pytz.timezone("America/New_York")

_FLIP = {"PAID": "RECEIVED", "RECEIVED": "PAID"}


def _resolve_tenor_dates(tenor_str, curve_id, as_of):
    """Resolve query tenor to (effective_date, maturity_date) pairs for trade matching."""
    from Query.IRSwaps._CENTRAL_BANK_DATES import resolve_central_bank_tenor

    if tenor_str is None:
        return []

    t = str(tenor_str)
    parts = [p.strip() for p in t.split("/")]
    date_ranges = []

    for part in parts:
        if part.startswith("fomc_"):
            dates = resolve_central_bank_tenor(curve_id, part, as_of=as_of)
            if dates and len(dates) >= 2:
                date_ranges.append((
                    pd.Timestamp(dates[0]).date(),
                    pd.Timestamp(dates[1]).date(),
                ))

    return date_ranges


def dealer_flow_chart(
    query,
    date,
    start_hour=8,
    end_hour=17,
    n_jobs=8,
    show_tqdm=True,
    customer=False,
    match_tenor=True,
):
    start = NY.localize(datetime.datetime(date.year, date.month, date.day, start_hour, 0))
    end = NY.localize(datetime.datetime(date.year, date.month, date.day, end_hour, 0))

    curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
    ts = TimeseriesBuilder()
    rate_df = ts.get_timeseries(
        start=start, end=end,
        queries=[query],
        freq="1min",
        n_jobs=n_jobs,
        routers={"IRS": IRSwapsTB(curve_mdp, show_tqdm=show_tqdm)},
    )
    if rate_df.empty:
        raise RuntimeError("No rate data returned")

    col = rate_df.columns[0]
    rate_series = rate_df[col].dropna()

    conn = psycopg2.connect(resolve_pg_url())

    # Pull direction + tape leg dates for matching
    directions = pd.read_sql(f"""
        SELECT d.unit_key, d.execution_timestamp, d.dealer_direction,
               d.classification_method, d.direction_confidence,
               d.structure_dv01, d.notional, d.fixed_rate, d.curve_mid,
               d.spread_to_mid_bps, d.dealer_charge_bps,
               d.rate_index_clean, d.trade_type, d.is_off_market,
               d.tenor_query,
               l.effective_date, l.expiration_date, l.tenor_label,
               l.special_tenor_type, l.fomc_meeting_label
        FROM arbs_stir_direction_v1 d
        LEFT JOIN arbs_usd_swap_tape_legs_v2 l
          ON d.trade_id = l.trade_id
          AND l.as_of_date = d.as_of_date
        WHERE d.execution_timestamp::date = '{date.isoformat()}'
          AND d.dealer_direction IN ('PAID', 'RECEIVED')
        ORDER BY d.execution_timestamp
    """, conn)
    conn.close()

    # Deduplicate: multi-leg packages join multiple legs per direction row
    directions = directions.drop_duplicates(subset=["unit_key"], keep="first")

    # Resolve query tenor to date ranges for matching
    tenor_str = getattr(query, 'tenor', None)
    date_ranges = _resolve_tenor_dates(tenor_str, "USD-OIS", date)

    if match_tenor and date_ranges:
        directions["effective_date"] = pd.to_datetime(directions["effective_date"]).dt.date
        directions["expiration_date"] = pd.to_datetime(directions["expiration_date"]).dt.date

        mask = pd.Series(False, index=directions.index)
        for eff, mat in date_ranges:
            mask |= (directions["effective_date"] <= eff) & (directions["expiration_date"] >= mat)

        n_before = len(directions)
        directions = directions[mask]
        label_parts = [f"{e.isoformat()}→{m.isoformat()}" for e, m in date_ranges]
        print(f"Tenor match: {len(directions)}/{n_before} trades span {', '.join(label_parts)}")

    directions["execution_timestamp"] = pd.to_datetime(directions["execution_timestamp"])
    if rate_series.index.tz is not None:
        if directions["execution_timestamp"].dt.tz is None:
            directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_localize("UTC")
        directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_convert(rate_series.index.tz)
    mask = (directions["execution_timestamp"] >= rate_series.index.min()) & \
           (directions["execution_timestamp"] <= rate_series.index.max())
    directions = directions[mask]

    if customer:
        directions["direction"] = directions["dealer_direction"].map(_FLIP)
    else:
        directions["direction"] = directions["dealer_direction"]

    perspective = "customer" if customer else "dealer"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=rate_series.index, y=rate_series.values,
        mode="lines", line=dict(width=2, color="#898781"),
        name=str(col), hovertemplate="%{y:.4f}<extra>rate</extra>",
    ))

    for direction, symbol, color in [
        ("RECEIVED", "triangle-up", "#008300"),
        ("PAID", "triangle-down", "#e34948"),
    ]:
        sub = directions[directions["direction"] == direction]
        if sub.empty:
            continue
        y_vals = []
        for t in sub["execution_timestamp"]:
            idx = rate_series.index.get_indexer([t], method="nearest")
            y_vals.append(float(rate_series.iloc[idx[0]]) if idx[0] != -1 else np.nan)

        sizes = np.clip(sub["structure_dv01"].fillna(0).values / 5000, 4, 25)

        eff_str = sub["effective_date"].astype(str).values
        exp_str = sub["expiration_date"].astype(str).values

        fig.add_trace(go.Scatter(
            x=sub["execution_timestamp"], y=y_vals,
            mode="markers",
            marker=dict(symbol=symbol, size=sizes, color=color,
                        line=dict(width=1, color="white")),
            name=direction,
            customdata=list(zip(
                sub["structure_dv01"].fillna(0).round(0).values,
                sub["notional"].fillna(0).apply(lambda x: f"{x/1e6:.1f}M").values,
                sub["fixed_rate"].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "?").values,
                sub["curve_mid"].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "?").values,
                sub["spread_to_mid_bps"].apply(lambda x: f"{x:+.1f}bp" if pd.notna(x) else "?").values,
                eff_str, exp_str,
                sub["tenor_label"].fillna("?").values,
                sub["direction_confidence"].fillna("?").values,
                sub["classification_method"].fillna("?").values,
                sub["trade_type"].fillna("?").values,
                sub["rate_index_clean"].fillna("?").values,
            )),
            hovertemplate=(
                f"<b>{perspective} {direction}</b><br>"
                "DV01: %{customdata[0]:,.0f}  notional: %{customdata[1]}<br>"
                "rate: %{customdata[2]}  mid: %{customdata[3]}  s2m: %{customdata[4]}<br>"
                "eff: %{customdata[5]}  exp: %{customdata[6]}  tenor: %{customdata[7]}<br>"
                "type: %{customdata[10]}  index: %{customdata[11]}<br>"
                "conf: %{customdata[8]}  method: %{customdata[9]}"
                "<extra></extra>"
            ),
        ))

    tenor_label = getattr(query, 'tenor', str(col))
    fig.update_layout(
        title=f"{tenor_label} — {date.isoformat()} {perspective} flow",
        yaxis_title="rate (bps)" if rate_series.max() < 1 else "rate (%)",
        xaxis_title="",
        height=500,
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="white",
        font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        yaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
        xaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
    )
    return fig

In [ ]:
q = UnifiedQuery(
    curve="USD-OIS-Q12xM12STIRT-SERFFX-MIX23",
    tenor="fomc_jul26",
    value=UnifiedValue.IRS_RATE,
)

fig = dealer_flow_chart(q, datetime.date(2026, 7, 2), customer=True)
fig.show()